In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_atraso")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3ffcd3e5-1afd-4bcf-a25b-34b18ee28fa2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 675ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/book_atraso/"
df_book_atraso = spark.read.parquet(path)
df_book_atraso.show(20, truncate=False)

26/01/02 19:18:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/01/02 19:18:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------+----------------------------------------------------------------+------------------+---------+-------------+------------------------+--------------+-------+--------+---------------------+---------+---------------------+---------------------+-------------------+------------------------+-------------------+--------------+--------------------------+----------------------------+--------------------+---------------------+----------------------+------------------+------------------+------------------+----------------------+----------------+------------------+-------------------+------+-------+--------+-------+----------------+----------+---------------+-------------+---------------+--------------+----------------+-----------------------+--------------+------------------+---------------+----------------------+---------------------+-----------------+----------------------+------------------+
|NUM_CPF    |DAT_REFERENCIA    |NUM_FATURA_HASH                        

In [4]:
df_book_atraso.createOrReplaceTempView("raw_00")

In [5]:
df_book_atraso.count()

31611316

In [6]:

raw_00_com_safra = spark.sql("""
    SELECT
        *,
        CAST(
            date_format(
                to_timestamp(DAT_REFERENCIA, 'ddMMMyyyy:HH:mm:ss'),
                'yyyyMM'
            ) AS INT
        ) AS SAFRA
    FROM raw_00
""")

raw_00_com_safra.createOrReplaceTempView("raw_00_com_safra")

In [7]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM raw_00_com_safra
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM raw_00_com_safra r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [8]:
df_resultado = contagem_percentual("DAT_ALTERACAO_VCTO_FAT")
df_resultado.show(truncate=False)

+------------------+-------------+-------------+
|valor_coluna      |qtd_registros|pct_registros|
+------------------+-------------+-------------+
|NULL              |31523457     |99.72        |
|14NOV2024:00:00:00|21087        |0.07         |
|22NOV2023:00:00:00|2520         |0.01         |
|12AUG2024:00:00:00|1666         |0.01         |
|10NOV2023:00:00:00|1238         |0.00         |
|12DEC2023:00:00:00|1137         |0.00         |
|27NOV2023:00:00:00|850          |0.00         |
|11OCT2023:00:00:00|840          |0.00         |
|13MAY2024:00:00:00|816          |0.00         |
|27AUG2018:00:00:00|810          |0.00         |
|20MAR2024:00:00:00|800          |0.00         |
|26OCT2023:00:00:00|789          |0.00         |
|27DEC2023:00:00:00|738          |0.00         |
|10JUL2024:00:00:00|639          |0.00         |
|09FEB2024:00:00:00|630          |0.00         |
|27JAN2025:00:00:00|620          |0.00         |
|25MAY2023:00:00:00|576          |0.00         |
|28OCT2024:00:00:00|

In [9]:
df_resultado = contagem_percentual("DAT_CANCELAMENTO_FAT")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|NULL        |31611316     |100.00       |
+------------+-------------+-------------+



In [10]:
df_resultado = contagem_percentual("COD_PLATAFORMA")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|AUTOC       |17230902     |54.51        |
|POSPG       |10963007     |34.68        |
|-2          |2271850      |7.19         |
|PREPG       |748188       |2.37         |
|POSBL       |240839       |0.76         |
|FLEXD       |63540        |0.20         |
|-3          |46901        |0.15         |
|CTLFC       |42179        |0.13         |
|POSTL       |2470         |0.01         |
|M2MS        |930          |0.00         |
|MVNOD       |324          |0.00         |
|POSRI       |105          |0.00         |
|POSCW       |51           |0.00         |
|SGIOT       |30           |0.00         |
+------------+-------------+-------------+



In [11]:
df_resultado = contagem_percentual("NUM_BILL_SEQ_FAT")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|1           |4082744      |12.92        |
|2           |2282121      |7.22         |
|3           |2111113      |6.68         |
|4           |1404737      |4.44         |
|5           |956813       |3.03         |
|6           |762806       |2.41         |
|7           |704599       |2.23         |
|8           |635146       |2.01         |
|9           |590163       |1.87         |
|10          |562407       |1.78         |
|11          |540528       |1.71         |
|12          |528790       |1.67         |
|13          |500384       |1.58         |
|14          |465651       |1.47         |
|15          |441262       |1.40         |
|16          |421552       |1.33         |
|17          |406469       |1.29         |
|18          |390371       |1.23         |
|19          |375906       |1.19         |
|25          |367049       |1.16         |
+----------

In [12]:
df_resultado = contagem_percentual("NUM_SEQ_ACORDO_FAT")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|-2          |27362822     |86.56        |
|-3          |3572475      |11.30        |
|1           |192679       |0.61         |
|2           |113230       |0.36         |
|3           |75216        |0.24         |
|4           |53651        |0.17         |
|5           |40730        |0.13         |
|6           |32699        |0.10         |
|7           |26156        |0.08         |
|8           |22270        |0.07         |
|9           |18478        |0.06         |
|10          |15591        |0.05         |
|11          |13336        |0.04         |
|12          |11049        |0.03         |
|13          |9492         |0.03         |
|14          |7741         |0.02         |
|15          |6636         |0.02         |
|16          |5524         |0.02         |
|17          |4763         |0.02         |
|18          |3987         |0.01         |
+----------

In [13]:
df_resultado = contagem_percentual("IND_ISENCAO_COB_FAT")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|-2          |28026258     |88.66        |
|N           |3580722      |11.33        |
|Y           |3442         |0.01         |
|S           |894          |0.00         |
+------------+-------------+-------------+



In [14]:
print('lista de colunas para tipar')
for col in spark.table("raw_00_com_safra").columns:
    print('try_cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
try_cast(NUM_CPF as) as NUM_CPF,
try_cast(DAT_REFERENCIA as) as DAT_REFERENCIA,
try_cast(NUM_FATURA_HASH as) as NUM_FATURA_HASH,
try_cast(NUM_ENT_SEQ_FATURA as) as NUM_ENT_SEQ_FATURA,
try_cast(CONTRATO as) as CONTRATO,
try_cast(DW_UN_NEGOCIO as) as DW_UN_NEGOCIO,
try_cast(DW_HIS_PONTO_VENDA_COMTA as) as DW_HIS_PONTO_VENDA_COMTA,
try_cast(DW_NUM_CLIENTE as) as DW_NUM_CLIENTE,
try_cast(DW_AREA as) as DW_AREA,
try_cast(DW_CICLO as) as DW_CICLO,
try_cast(DW_TIPO_CLIENTE_CONTA as) as DW_TIPO_CLIENTE_CONTA,
try_cast(DW_OFERTA as) as DW_OFERTA,
try_cast(DW_FAIXA_AGING_FATURA as) as DW_FAIXA_AGING_FATURA,
try_cast(DW_FAIXA_AGING_DIVIDA as) as DW_FAIXA_AGING_DIVIDA,
try_cast(DW_FAIXA_TEMPO_BASE as) as DW_FAIXA_TEMPO_BASE,
try_cast(DW_FAIXA_AGING_PROX_FECH as) as DW_FAIXA_AGING_PROX_FECH,
try_cast(DW_TIPO_FATURAMENTO as) as DW_TIPO_FATURAMENTO,
try_cast(COD_PLATAFORMA as) as COD_PLATAFORMA,
try_cast(DAT_CRIACAO_REGISTRO_TRANS as) as DAT_CRIACAO_REGISTRO_TRANS,
try_cas

In [15]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            case 
                when trim(DAT_REFERENCIA) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_REFERENCIA), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_REFERENCIA,
            try_cast(NUM_FATURA_HASH as STRING) as NUM_FATURA_HASH,
            try_cast(NUM_ENT_SEQ_FATURA as INT) as NUM_ENT_SEQ_FATURA,
            try_cast(CONTRATO as BIGINT) as CONTRATO,
            try_cast(DW_UN_NEGOCIO as INT) as DW_UN_NEGOCIO,
            try_cast(DW_HIS_PONTO_VENDA_COMTA as BIGINT) as DW_HIS_PONTO_VENDA_COMTA,
            try_cast(DW_NUM_CLIENTE as STRING) as DW_NUM_CLIENTE,
            try_cast(DW_AREA as INT) as DW_AREA,
            try_cast(DW_CICLO as INT) as DW_CICLO,
            try_cast(DW_TIPO_CLIENTE_CONTA as INT) as DW_TIPO_CLIENTE_CONTA,
            try_cast(DW_OFERTA as INT) as DW_OFERTA,
            try_cast(DW_FAIXA_AGING_FATURA as INT) as DW_FAIXA_AGING_FATURA,
            try_cast(DW_FAIXA_AGING_DIVIDA as INT) as DW_FAIXA_AGING_DIVIDA,
            try_cast(DW_FAIXA_TEMPO_BASE as INT) as DW_FAIXA_TEMPO_BASE,
            try_cast(DW_FAIXA_AGING_PROX_FECH as INT) as DW_FAIXA_AGING_PROX_FECH,
            try_cast(DW_TIPO_FATURAMENTO as INT) as DW_TIPO_FATURAMENTO,
            try_cast(COD_PLATAFORMA as STRING) as COD_PLATAFORMA,
            case 
                when trim(DAT_CRIACAO_REGISTRO_TRANS) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_REGISTRO_TRANS), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_REGISTRO_TRANS,
            case 
                when trim(DAT_ALTERACAO_REGISTRO_TRANS) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ALTERACAO_REGISTRO_TRANS), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ALTERACAO_REGISTRO_TRANS,
            case 
                when trim(DAT_CANCELAMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CANCELAMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CANCELAMENTO_FAT,
            case 
                when trim(DAT_ORIGINAL_VCTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ORIGINAL_VCTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ORIGINAL_VCTO_FAT,
            case 
                when trim(DAT_ALTERACAO_VCTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ALTERACAO_VCTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ALTERACAO_VCTO_FAT,
            case 
                when trim(DAT_CRIACAO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_FAT,
            case 
                when trim(DAT_VENCIMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_VENCIMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_VENCIMENTO_FAT,
            case 
                when trim(DAT_STATUS_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_FAT,
            case 
                when trim(DAT_MIN_VENCIMENTO_FAT) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_MIN_VENCIMENTO_FAT), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_MIN_VENCIMENTO_FAT,
            try_cast(NUM_BILL_SEQ_FAT as INT) as NUM_BILL_SEQ_FAT,
            try_cast(NUM_SEQ_ACORDO_FAT as INT) as NUM_SEQ_ACORDO_FAT,
            try_cast(IND_ISENCAO_COB_FAT as STRING) as IND_ISENCAO_COB_FAT,
            try_cast(IND_WO as STRING) as IND_WO,
            try_cast(IND_PDD as STRING) as IND_PDD,
            try_cast(IND_PCCR as STRING) as IND_PCCR,
            try_cast(IND_ACA as STRING) as IND_ACA,
            try_cast(IND_PRIMEIRA_FAT as STRING) as IND_PRIMEIRA_FAT,
            try_cast(IND_FRAUDE as STRING) as IND_FRAUDE,
            try_cast(VAL_FAT_LIQUIDO as DECIMAL(10,2)) as VAL_FAT_LIQUIDO,
            try_cast(VAL_FAT_BRUTO as DECIMAL(10,2)) as VAL_FAT_BRUTO,
            try_cast(VAL_FAT_CREDITO as DECIMAL(10,2)) as VAL_FAT_CREDITO,
            try_cast(VAL_FAT_AJUSTE as DECIMAL(10,2)) as VAL_FAT_AJUSTE,
            try_cast(VAL_FAT_BRUTO_BC as DECIMAL(10,2)) as VAL_FAT_BRUTO_BC,
            try_cast(VAL_FAT_PAGAMENTO_BRUTO as DECIMAL(10,2)) as VAL_FAT_PAGAMENTO_BRUTO,
            try_cast(VAL_FAT_ABERTO as DECIMAL(10,2)) as VAL_FAT_ABERTO,
            try_cast(VAL_FAT_ABERTO_LIQ as DECIMAL(10,2)) as VAL_FAT_ABERTO_LIQ,
            try_cast(VAL_MULTA_JUROS as DECIMAL(10,2)) as VAL_MULTA_JUROS,
            try_cast(VAL_MULTA_CANCELAMENTO as DECIMAL(10,2)) as VAL_MULTA_CANCELAMENTO,
            try_cast(VAL_PARC_APARELHO_LIQ as DECIMAL(10,2)) as VAL_PARC_APARELHO_LIQ,
            try_cast(VAL_FAT_LIQ_JM_MC as DECIMAL(10,2)) as VAL_FAT_LIQ_JM_MC,
            case 
                when trim(DAT_ATIVACAO_CONTA_CLI) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_ATIVACAO_CONTA_CLI), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATIVACAO_CONTA_CLI,
            case 
                when trim(DAT_CRIACAO_DW) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_DW), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_DW,
            {pdthproc} as DATPROC

        from
            raw_00_com_safra
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

31611316

In [16]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- DAT_REFERENCIA: timestamp (nullable = true)
 |-- NUM_FATURA_HASH: string (nullable = true)
 |-- NUM_ENT_SEQ_FATURA: integer (nullable = true)
 |-- CONTRATO: long (nullable = true)
 |-- DW_UN_NEGOCIO: integer (nullable = true)
 |-- DW_HIS_PONTO_VENDA_COMTA: long (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: integer (nullable = true)
 |-- DW_CICLO: integer (nullable = true)
 |-- DW_TIPO_CLIENTE_CONTA: integer (nullable = true)
 |-- DW_OFERTA: integer (nullable = true)
 |-- DW_FAIXA_AGING_FATURA: integer (nullable = true)
 |-- DW_FAIXA_AGING_DIVIDA: integer (nullable = true)
 |-- DW_FAIXA_TEMPO_BASE: integer (nullable = true)
 |-- DW_FAIXA_AGING_PROX_FECH: integer (nullable = true)
 |-- DW_TIPO_FATURAMENTO: integer (nullable = true)
 |-- COD_PLATAFORMA: string (nullable = true)
 |-- DAT_CRIACAO_REGISTRO_TRANS: timestamp (nullable = true)
 |-- DAT_ALTERACAO_REGISTRO_TRANS:

In [17]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, DAT_REFERENCIA, CONTRATO, NUM_ENT_SEQ_FATURA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

31611219

In [18]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_atraso/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.DAT_REFERENCIA = s.DAT_REFERENCIA
            AND t.CONTRATO = s.CONTRATO
            AND t.NUM_ENT_SEQ_FATURA = s.NUM_ENT_SEQ_FATURA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")

Tabela silver não existe. Criando...


Dados inseridos com sucesso...


In [22]:
name = "base_atraso"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""".format(name_table=name))

df_controle.show()

Py4JJavaError: An error occurred while calling o106.showString.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(Unknown Source)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(Unknown Source)
java.base/java.lang.reflect.Constructor.newInstance(Unknown Source)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
py4j

In [20]:
silver_controle_path = "s3a://silver/controle/"
if not DeltaTable.isDeltaTable(spark, silver_controle_path):
    print("Tabela de controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela de controle existe. Inserindo novo registro...")

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")

Tabela de controle existe. Inserindo novo registro...


Dados inseridos com sucesso...


In [21]:
spark.stop()